t-Testing

In [58]:
import pandas as pd
import numpy as np
from scipy import stats

Load the Data

In [59]:
df = pd.read_csv('../../output/processed_data/05_filtered.csv')

Daily Abnormal Returns

In [60]:
# AR was computed and sanity-checked in NB05.
# It is loaded directly from 05_filtered.csv — no need to recompute here.
assert 'AR' in df.columns, \
    "AR column missing from 05_filtered.csv — re-run NB05 first."
print(f'✓ AR loaded from 05_filtered.csv  ({df["AR"].notna().sum()} non-null values)')

✓ AR loaded from 05_filtered.csv  (69390 non-null values)


Event Window

In [61]:
windows = {
    'CAR(-5,-1)': (-5, -1),
    'CAR(0,0)': (0, 0),
    'CAR(0,+5)': (0, 5),
    'CAR(-5,+30)': (-5, 30)
}

t-test Function

In [62]:
def run_cross_sectional_ttest(data, start_day, end_day, window_name):
    #Filter the data
    window_data = data[(data['day'] >= start_day) & (data['day'] <= end_day)]

    #sum daily ARto get CAR
    stock_cars = window_data.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

    results = []

    # Sector level t-test
    sectors = stock_cars['sector'].unique()

    for sector in sectors:
        sector_data = stock_cars[stock_cars['sector'] == sector]['CAR']
        n_stocks = len(sector_data)
        
        if n_stocks > 1:
            mean_car = sector_data.mean()
            std_car = sector_data.std()
            t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
        else:
            mean_car = sector_data.mean()
            std_car = np.nan
            t_stat = np.nan
            p_val = np.nan

        results.append({
            'Window': window_name,
            'Level': 'Sector',
            'Name': sector,
            'N': n_stocks,
            'Mean_CAR(%)': mean_car * 100, # Convert to percentage for readability
            'Std_Dev(%)': std_car * 100,
            't-statistic': t_stat,
            'p-value': p_val,
            'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
        })

    #Market t-test
    all_cars = stock_cars['CAR']
    n_total = len(all_cars)
    t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_cars, popmean=0)

    results.append({
        'Window': window_name,
        'Level': 'Market',
        'Name': 'ALL STOCKS',
        'N': n_total,
        'Mean_CAR(%)': all_cars.mean() * 100, 
        'Std_Dev(%)': all_cars.std() * 100,
        't-statistic': t_stat_mkt,
        'p-value': p_val_mkt,
        'Significant_at_5%': p_val_mkt < 0.05 
    })

    return pd.DataFrame(results)



Test all windows

In [63]:
all_results = []

for window_name, (start, end) in windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    all_results.append(window_results_df)

final_ttest_results = pd.concat(all_results,ignore_index=True)

formatted_results = final_ttest_results.copy()
formatted_results['Mean_CAR(%)'] = formatted_results['Mean_CAR(%)'].round(2)
formatted_results['Std_Dev(%)'] = formatted_results['Std_Dev(%)'].round(2)
formatted_results['t-statistic'] = formatted_results['t-statistic'].round(3)
formatted_results['p-value'] = formatted_results['p-value'].round(4)

RQ3: Anticipation effect — did the market react before landfall?

In [64]:
anticipation = final_ttest_results[
    (final_ttest_results['Window'] == 'CAR(-5,-1)') &
    (final_ttest_results['Level']  == 'Market')
].iloc[0]

print('RQ3 — Anticipation Effect: CAR(-5,-1) market-wide')
print(f'  Mean CAR   : {anticipation["Mean_CAR(%)"]:.3f}%')
print(f'  t-statistic: {anticipation["t-statistic"]:.3f}')
print(f'  p-value    : {anticipation["p-value"]:.4f}')
print()
if anticipation['p-value'] < 0.05:
    print('  ✓ Statistically significant anticipation effect detected.')
    print('    The market was already pricing in disaster risk before landfall.')
else:
    print('  ✗ No statistically significant anticipation effect detected.')
print()

# Which sectors anticipated earliest?
print('Sector-level anticipation — CAR(-5,-1) sorted by mean CAR:')
sector_anticipation = (
    final_ttest_results[
        (final_ttest_results['Window'] == 'CAR(-5,-1)') &
        (final_ttest_results['Level']  == 'Sector')
    ]
    [['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']]
    .sort_values('Mean_CAR(%)')
)
print(sector_anticipation.to_string(index=False))

RQ3 — Anticipation Effect: CAR(-5,-1) market-wide
  Mean CAR   : -1.014%
  t-statistic: -2.955
  p-value    : 0.0034

  ✓ Statistically significant anticipation effect detected.
    The market was already pricing in disaster risk before landfall.

Sector-level anticipation — CAR(-5,-1) sorted by mean CAR:
                      Name  N  Mean_CAR(%)  t-statistic  p-value  Significant_at_5%
            Transportation  3    -6.410697    -7.014882 0.019722               True
                    Energy  3    -6.189744    -1.882075 0.200542              False
                 Retailing 14    -4.100072    -1.290609 0.219321              False
Telecommunication Services  2    -3.033591  -170.101306 0.003743               True
                 Materials 22    -2.510958    -1.778909 0.089735              False
         Consumer Durables  8    -2.061963    -1.113570 0.302235              False
                 Utilities  9    -1.825882    -1.730685 0.121755              False
       Software & Ser

In [65]:
# ── Recovery evidence — CAR(-5,+30) significance per sector ──────────────────
# Pairs with the recovery_time.csv from NB05.
# Sectors with p < 0.05 have statistically confirmed persistent damage.
# Sectors with p > 0.05 did not sustain significant full-window losses —
# consistent with rapid recovery.

assert 'final_ttest_results' in dir(), \
    "Run the 'Test all windows' cell above first before running this cell."

print('Recovery evidence — CAR(-5,+30) significance by sector:')
print('Significant (p<0.05) = statistically confirmed persistent damage\n')

rec_ev = (
    final_ttest_results[
        (final_ttest_results['Window'] == 'CAR(-5,+30)') &
        (final_ttest_results['Level']  == 'Sector')
    ]
    [['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']]
    .sort_values('Mean_CAR(%)')
    .reset_index(drop=True)
)
print(rec_ev.to_string(index=False))

rec_ev.to_csv('../../output/tables/recovery_evidence.csv', index=False)
print('\n✓ Saved recovery_evidence.csv')

Recovery evidence — CAR(-5,+30) significance by sector:
Significant (p<0.05) = statistically confirmed persistent damage

                      Name  N  Mean_CAR(%)  t-statistic      p-value  Significant_at_5%
            Transportation  3   -24.685264    -1.690458 2.330071e-01              False
                    Energy  3   -17.926195    -1.187104 3.570732e-01              False
                 Retailing 14   -16.582242    -0.779197 4.498161e-01              False
                 Insurance 13   -13.091825    -4.618328 5.918965e-04               True
                    Others  1   -13.084414          NaN          NaN              False
                 Utilities  9   -11.665490    -2.699728 2.708543e-02               True
         Consumer Services 33   -10.186320    -6.825601 1.017997e-07               True
    Diversified Financials 39    -9.552391    -4.436213 7.587958e-05               True
                 Materials 22    -4.899854    -0.688065 4.989422e-01              Fals

Results

In [66]:
display_df = formatted_results[formatted_results['Window'] == 'CAR(-5,+30)'].sort_values(by='Mean_CAR(%)')
print(display_df[['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']])


formatted_results.to_csv('../../output/tables/ttest_results.csv', index=False)

                          Name    N  Mean_CAR(%)  t-statistic  p-value  \
81              Transportation    3       -24.69       -1.690   0.2330   
85                      Energy    3       -17.93       -1.187   0.3571   
76                   Retailing   14       -16.58       -0.779   0.4498   
67                   Insurance   13       -13.09       -4.618   0.0006   
77                      Others    1       -13.08          NaN      NaN   
84                   Utilities    9       -11.67       -2.700   0.0271   
73           Consumer Services   33       -10.19       -6.826   0.0000   
66      Diversified Financials   39        -9.55       -4.436   0.0001   
87                  ALL STOCKS  279        -5.86       -4.083   0.0001   
71                   Materials   22        -4.90       -0.688   0.4989   
75                 Real Estate   17        -4.69       -1.181   0.2549   
78          Household Products    1        -4.36          NaN      NaN   
80              Food Retailing    4   